# ReBRAC Stage D Phase 1 — worldcomp epoch probe

**目的**：在启动 Stage D Phase 1 deployable formal 之前，确认 `TRAIN_EPOCHS` 在 `worldcomp-1000` 数据上的合适值。`crosscomp` 上 Stage B0 已经验证过 `TRAIN_EPOCHS=64` 够用，但 `worldcomp` 数据的瓶颈性质不同（teacher 不是 deployable，critic penalty 的 next_actions target 可能更 noisy），有必要在 worldcomp 数据上单独验证。

**方法**：复用 Stage B0 的 "长训读中间 ckpt" 思路——训一次 128 epoch，每 8 epoch 保存 ckpt，每个 ckpt 上跑 val 拿到 16 个 val 点。**只跑 deployable 轨道**，因为 epoch 选择是为了让训练充分（不依赖于 actor 是否看到 privileged obs）。

**实验 scope**：

| 轴 | 配置 | 说明 |
|---|---|---|
| β1 / β2 | `4.0 / 2.0` | Stage C 锁定的 finalist，Stage D 不再扫超参 |
| dataset | `worldcomp-1000` | Stage D 的核心 dataset |
| seeds | `42 43` | 只用 2 seed；epoch peak 不需要精确点估，需要的是 "≤64 / ∈(64,96] / >96" 三档判定 |
| TRAIN_EPOCHS | `128` | 产出 16 个 ckpt（每 8 epoch 一个） |
| manifest | val=40 / test=40 | 与 Stage B0 协议一致；Stage D formal 才升 test=100 |
| 轨道 | deployable | privileged-critic 不在 epoch probe 阶段做（参见 plan §6.5.2 Step 1） |

**预算**：1 cell × 1 dataset × 2 seed = 2 个 run，约 `1/3` 个 deployable formal 的代价。

**输出树**：刻意与 Stage D formal 隔离，避免 ckpt / 结果交叉污染：
- `checkpoints/offline/rebrac/worldcomp_epoch_probe/`
- `results/offline/rebrac/worldcomp_epoch_probe/`

**判据**（驱动 Stage D Phase 1 deployable formal 的 `TRAIN_EPOCHS` 决策）：

| 观察 | 决策 |
|---|---|
| 两 seed 的 val peak epoch 都 ≤ 64 | Stage D Phase 1 取 `TRAIN_EPOCHS=64` |
| 任一 seed 的 peak epoch ∈ (64, 96] | Stage D Phase 1 取 `TRAIN_EPOCHS=96`（对齐 TD3BC worldcomp teacher-gap 预算） |
| 任一 seed 的 peak epoch > 96 | Stage D Phase 1 暂停，回查 worldcomp 训练动力学（critic penalty 与 noisy teacher target 的相互作用） |


## 0. 环境配置


In [ ]:
import os

# —— 通用（与其它 ReBRAC / TD3BC 实验一致）——
os.environ["PYTHON_BIN"]         = "python3"
os.environ["DEVICE"]             = "cuda"
os.environ["EVAL_WORKERS"]       = "6"
os.environ["EVAL_WORKER_DEVICE"] = "cpu"

# —— Stage D 在 worldcomp 上跑，覆盖 driver 默认（默认是 worldcomp）——
# DATASET_POLICY=worldcomp 已经是 teacher-gap driver 默认，这里不需要再设。

# —— Phase 1 epoch probe 网格点（覆盖 driver 默认）——
os.environ["EPOCH_PROBE_SEEDS"]                       = "42 43"
os.environ["EPOCH_PROBE_TRAIN_EPOCHS"]                = "128"
os.environ["EPOCH_PROBE_CHECKPOINT_EVERY_EPOCHS"]     = "8"
os.environ["EPOCH_PROBE_VAL_MANIFEST_EPISODES"]       = "40"
os.environ["EPOCH_PROBE_TEST_MANIFEST_EPISODES"]      = "40"

# —— Stage C 锁定的 finalist（Stage D 不扫超参）——
os.environ["ACTOR_PENALTY_COEF"]  = "4.0"
os.environ["CRITIC_PENALTY_COEF"] = "2.0"

# 其余（DATASET_POLICY=worldcomp, DATASET_EPISODES_VALUE=1000,
# BENCHMARK_KEY=single_u10_cross_tgt15, PROBE_LAYOUT=s0,
# HISTORY_LENGTH=4, TASK_GEOMETRY=cross_stream, TARGET_SPEED=1.5,
# OBJECTIVE=efficiency_v2, SAMPLING_MODE=shuffle_no_replacement,
# BATCH_SIZE=256）使用 driver 默认值。


## 1. 生成 val / test manifest

只跑一次。已经存在的文件会被跳过。


In [ ]:
os.environ["MODE"] = "epoch_probe_manifests"
!bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh


## 2. 收集 worldcomp-1000 离线数据

如果之前 TD3BC worldcomp teacher-gap 已经跑过 collect 生成了 `offline_data/worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000`，这步会打印 `[skip] dataset exists` 并跳过。否则会用 `worldcomp` baseline policy 跑 1000 episodes 的数据收集。


In [ ]:
os.environ["MODE"] = "epoch_probe_collect"
!bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh


## 3. 全流程：train → validate（每个 ckpt）→ select → test → summarize

这一步产出 2 个 run，每个 run 16 个 ckpt，每个 ckpt 在 val manifest 上评估 1 次。耗时主体是 train（128 epoch × 2 seed）。


In [ ]:
# 跑全部 epoch_probe 流程：train + validate + test + summarize
for mode in ["epoch_probe_train", "epoch_probe_validate", "epoch_probe_test", "epoch_probe_summarize"]:
    os.environ["MODE"] = mode
    !bash scripts/run_offline_rebrac_worldcomp_teacher_gap.sh


## 4. 分析：val 曲线 vs epoch

`select_best_checkpoint`（screen.sh 内部调用）写出的 `selected_checkpoint.json` 里有 `candidates` 数组，每个元素就是一个 `(train_step, eval_success_rate, eval_return, …)` 点——现成的 val 曲线。

把 `train_step` 换算成 epoch：最后一个 ckpt 对应 `TRAIN_EPOCHS=128`，所以 `epoch = train_step / (max_train_step / 128)`。


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

RESULTS_ROOT     = Path("results/offline/rebrac/worldcomp_epoch_probe")
TRAIN_EPOCHS     = int(os.environ["EPOCH_PROBE_TRAIN_EPOCHS"])
PAIR_TAG         = "actorb_4p0__criticb_2p0"
DATASET_NAME     = "worldcomp_s0_h4_efficiency_v2_re150_u10cross_fixdone_ep1000"
SEEDS            = os.environ["EPOCH_PROBE_SEEDS"].split()
EPOCH_THRESHOLDS = [64, 96]  # 决策三档分界


def load_candidates(seed: str) -> list[dict]:
    selection_path = (
        RESULTS_ROOT / DATASET_NAME / PAIR_TAG / "selection" / f"seed_{seed}" / "selected_checkpoint.json"
    )
    if not selection_path.exists():
        print(f"[warn] missing: {selection_path}")
        return []
    payload = json.loads(selection_path.read_text(encoding="utf-8"))
    cands = [c for c in payload["candidates"] if c.get("train_step") is not None]
    cands.sort(key=lambda c: c["train_step"])
    return cands


def to_epoch(candidates: list[dict], train_epochs: int) -> list[float]:
    max_step = max(c["train_step"] for c in candidates)
    steps_per_epoch = max_step / train_epochs
    return [c["train_step"] / steps_per_epoch for c in candidates]


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))

for seed in SEEDS:
    cands = load_candidates(seed)
    if not cands:
        continue
    epochs = to_epoch(cands, TRAIN_EPOCHS)
    succ   = [c["eval_success_rate"] for c in cands]
    ax.plot(epochs, succ, marker="o", alpha=0.8, label=f"seed_{seed}")

for thr in EPOCH_THRESHOLDS:
    ax.axvline(thr, linestyle="--", color="gray", alpha=0.5, label=f"decision boundary ({thr})")

ax.set_title(f"ReBRAC val curve vs training budget on {DATASET_NAME}\n"
             f"(β1={os.environ['ACTOR_PENALTY_COEF']}, β2={os.environ['CRITIC_PENALTY_COEF']}, deployable track)")
ax.set_xlabel("epoch")
ax.set_ylabel("val success_rate")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()


In [ ]:
# Peak epoch 汇总表 + 决策建议
print(f"{'seed':<8}{'peak_epoch':<14}{'peak_success':<16}{'succ@ep64':<14}{'succ@ep96':<14}")
print("-" * 70)

peak_epochs = []
for seed in SEEDS:
    cands = load_candidates(seed)
    if not cands:
        continue
    epochs = to_epoch(cands, TRAIN_EPOCHS)
    succ   = [c["eval_success_rate"] for c in cands]
    peak_idx  = max(range(len(succ)), key=lambda i: succ[i])
    peak_ep   = epochs[peak_idx]
    peak_succ = succ[peak_idx]
    ref64_idx = min(range(len(epochs)), key=lambda i: abs(epochs[i] - 64))
    ref96_idx = min(range(len(epochs)), key=lambda i: abs(epochs[i] - 96))
    print(f"{seed:<8}{peak_ep:<14.1f}{peak_succ:<16.4f}{succ[ref64_idx]:<14.4f}{succ[ref96_idx]:<14.4f}")
    peak_epochs.append(peak_ep)

print()
if peak_epochs:
    max_peak = max(peak_epochs)
    if max_peak <= 64:
        decision = "TRAIN_EPOCHS=64 (Phase 1 deployable formal 用 64)"
    elif max_peak <= 96:
        decision = "TRAIN_EPOCHS=96 (对齐 TD3BC worldcomp teacher-gap 的预算)"
    else:
        decision = "暂停 Phase 1，回查 worldcomp 训练动力学（critic penalty 与 noisy teacher target 的相互作用）"
    print(f"[decision] max peak epoch = {max_peak:.1f} → Stage D Phase 1: {decision}")


## 5. 决策记录

跑完上面 cell 后，把 `[decision]` 那一行的内容写入 `docs/rebrac_experiment_report.md` 的 Stage D 章节（Phase 1 epoch-probe 小节），并据此设置 Stage D Phase 1 deployable formal 的 `DEPLOYABLE_FINAL_TRAIN_EPOCHS` 环境变量（在下一个 notebook `rebrac_worldcomp_teacher_gap.ipynb` 的开头）。

## 6. （可选）Stage B0 等价性 sanity check

Stage B0 在 `crosscomp` 上做了 "长训中间 ckpt ≈ 单独训到该 epoch" 的间接验证；如果想在 `worldcomp` 上也做一次严格的 `state_dict_l2_distance` 交叉比对，可在 Phase 1 deployable formal 跑完后比较：

- `checkpoints/offline/rebrac/worldcomp_epoch_probe/.../seed_42/agent_step_<step_at_ep64>.pt`
- `checkpoints/offline/rebrac/worldcomp_teacher_gap/deployable/.../seed_42/agent_final.pt`（如果 Phase 1 用 64 epoch）

理论上两者权重应该完全一致（同 RNG seed、同 sampling、同 LR schedule）。这一项不是必须，但能闭环 [report §9 局限性](../docs/rebrac_experiment_report.md) 第 1 项。
